In [ ]:
products = pd.read_csv("../data/books.csv")
[p.id for _, p in products.iterrows()]

[0        0
 1        1
 2        2
 3        3
 4        4
       ... 
 995    995
 996    996
 997    997
 998    998
 999    999
 Name: id, Length: 1000, dtype: int64]

In [48]:
import logging
import os

import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import MinMaxScaler
from odoo import api, models

_logger = logging.getLogger(__name__)

_MODELS_DIR = "../addons/bookstore_recommendation_engine/static/models"
_ITEM_MODEL_PATH = os.path.join(_MODELS_DIR, "item_similarity.json")


class ItemRecommendationEngine():
    _name = "recommendation.engine.item"
    _inherit = "recommendation.engine.base"
    _description = "Item-Based Recommendation Engine"

    @api.model
    def _train_item_model(self, months=None, min_product_orders=1):

        products = pd.read_csv("../data/books.csv")

        if len(products) < 2:
            _logger.warning("Not enough products to train item model.")
            return False

        features = (
            products.set_index("id")
            .pipe(
                lambda df: pd.concat(
                    [
                        pd.get_dummies(
                            df[
                                [
                                    "genre",
                                    "sub_genre",
                                    "author",
                                    "publisher",
                                ]
                            ]
                        ),
                        pd.DataFrame(
                            MinMaxScaler().fit_transform(
                                df[
                                    [
                                        "year",
                                        "pages",
                                    ]
                                ]
                            ),
                            columns=["year", "pages"],
                            index=df.index,
                        ),
                    ],
                    axis=1,
                )
            )
        )

        similarity_df = pd.DataFrame(
            cosine_similarity(features.values),
            index=features.index,
            columns=features.index,
        )

        os.makedirs(_MODELS_DIR, exist_ok=True)
        similarity_df.to_json(
            _ITEM_MODEL_PATH,
            orient="records",
            indent=4,
        )

        _logger.info("Item model trained: %d products.", len(products))
        return True

    @api.model
    def _load_item_model(self):
        try:
            df = pd.read_json(_ITEM_MODEL_PATH, orient="records")
            df.index = df.index.astype(int)
            df.columns = df.columns.astype(int)
            return df
        except Exception as exc:
            _logger.error("Failed to load item model: %s", exc)
            return None

    @api.model
    def get_similar_products(
        self,
        product_tmpl_id,
        limit=6,
    ):
        similarity_df = self._load_item_model()
        if similarity_df is None or product_tmpl_id not in similarity_df.index:
            return self.env["product.template"]

        #source_sub_genre_id = pd.read_csv("../data/books.csv").set_index("id").loc[product_tmpl_id, "sub_genre_id"]

        scores = (
            (
                # Get similarity scores for the given product template, excluding itself
                similarity_df[product_tmpl_id]
                .drop(product_tmpl_id)
                .nlargest(limit * 3)
                .to_frame("similarity")
                # Add product template records to apply additional filters and boosts
                .assign(
                    product_card=lambda df: pd.read_csv("../data/books.csv").set_index("id").loc[df.index].to_dict("index")
                )
                # Boost similarity for products in the same sub-genre and promoted books
                # .assign(
                #     similarity=lambda df: df.similarity
                #     + df.product_card.apply(lambda p: p.book_sub_genre_id.id).eq(
                #         source_sub_genre_id
                #     )
                #     * 0.2
                # )
            )
            .sort_values(
                "similarity",
                ascending=False,
            )
            .head(limit)
        )

        return scores


In [49]:
engine = ItemRecommendationEngine()
engine._train_item_model()

True

In [50]:
harry_potter = pd.read_csv("../data/books.csv").iloc[107]
harry_potter

id                                                            107
title                Harry Potter and the Deathly Hallows, Book 7
author                                               J.K. Rowling
isbn                                                9785891783909
parent_genre                               Children & Young Adult
genre                                            Children's Books
sub_genre                              Growing Up & Facts of Life
price                                                       17.59
pages                                                         871
publisher                                              Scholastic
year                                                         1981
description     Un viaje apasionante a través de ideas que tra...
Name: 107, dtype: object

In [51]:
engine.get_similar_products(harry_potter.id)

,similarity,product_card
618,0.789942,"{'title': 'Matilda: Special Edition', 'author'..."
446,0.784442,{'title': 'Diary of a Wimpy Kid 8 : Hard Luck'...
209,0.780179,{'title': 'HARRY POTTER AND THE ORDER OF THE P...
183,0.779736,{'title': 'Harry Potter and the Half-Blood Pri...
74,0.779491,{'title': 'Harry Potter and the Philosopher's ...
185,0.767371,{'title': 'Harry Potter and the Half-Blood Pri...
